# Trustworthy CXR VLM — Phase III (v3): balanced LoRA fine-tuning with generation-based selection (FP16)

The v2 adapter collapsed to one normal template (DR-031). v3 (DR-032) trains on the **train split only** with
rare-finding oversampling and a trainable projector, and picks the checkpoint by CheXbert findings-F1 of greedy
S1 reports on 100 val patients. The zero-shot model is a candidate: if it wins, no v3 generation is run.
Otherwise S0 and S1 (+5 samples for S3) are regenerated for val and test and scored into `results_v3/`.

**Before running:** Accelerator **GPU T4 x2**, Internet **On**, add the IU X-Ray dataset as an input.

Generation stops cleanly at `GEN_BUDGET_H`. If the log says **GENERATION NOT FINISHED**, attach this
version's output as an input and Save & Run All again: the selection log, adapter and finished patients are
reused. Download `phase3_v3_results.zip` (top level of the output) from the version whose log says
**GENERATION COMPLETE**.

In [ ]:
import glob, os, time
START = time.time()
GEN_BUDGET_H = 10.5          # stop starting new patients after this many hours (training time included)
REPO_URL = "https://github.com/justinthomas11/mts-medical-vlm.git"
REPO_COMMIT = None
IU_DATASET_DIR = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
print("IU_DATASET_DIR =", IU_DATASET_DIR)
QUANT = "fp16"
WORK = "/kaggle/working/mts-medical-vlm"
RESULTS = "results_v3"
ADAPTER = "checkpoints/llava_med_lora_v3"

In [ ]:
import os, subprocess, sys
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", REPO_URL, WORK], check=True)
if REPO_COMMIT:
    subprocess.run(["git", "-C", WORK, "checkout", REPO_COMMIT], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Same transformers version as local development (DR-022); fp16 needs no bitsandbytes.
# Scoring dependencies are installed up front so a broken install fails now, not after generation.
!pip install -q "transformers==5.17.0" accelerate "peft==0.22.0" "torchxrayvision==1.5.4" "pycocoevalcap==1.2"
!pip install -q --no-deps "f1chexbert==0.0.2" "appdirs==1.4.4"
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.device_count(), "GPU(s)",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

# RadGraph's bundled AllenNLP code needs transformers < 5, so it gets its own venv (DR-022).
RG_VENV = "/tmp/radgraph-venv"
RG_PY = os.path.join(RG_VENV, "bin", "python")
if not os.path.exists(RG_PY):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", "-q", RG_VENV], check=True)
    subprocess.run([RG_PY, "-m", "pip", "install", "-q", "torch==2.14.0",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run([RG_PY, "-m", "pip", "install", "-q", "-r", "requirements-radgraph.txt"], check=True)
subprocess.run([RG_PY, "-c", "import radgraph, transformers; print('radgraph venv ok, transformers', transformers.__version__)"],
               check=True)
os.environ["RADGRAPH_PYTHON"] = RG_PY

In [ ]:
# Link the raw IU X-Ray files where configs/data_config.yaml expects them (data/raw is git-ignored).
os.makedirs("data/raw/images", exist_ok=True)
for src, dst in [("indiana_reports.csv", "data/raw/indiana_reports.csv"),
                 ("indiana_projections.csv", "data/raw/indiana_projections.csv"),
                 ("images/images_normalized", "data/raw/images/images_normalized")]:
    if not os.path.exists(dst):
        os.symlink(os.path.join(IU_DATASET_DIR, src), dst)
print(len(os.listdir("data/raw/images/images_normalized")), "images linked (expected 7,470)")

In [ ]:
# Rebuild data/processed with the fixed seed, then STOP unless the split is identical to the committed one.
subprocess.run([sys.executable, "src/data/split_dataset.py"], check=True)
check = subprocess.run([sys.executable, "src/pipeline/verify_splits.py"])
assert check.returncode == 0, "Split differs from the committed benchmark — do not run any stage."

In [ ]:
# Full test suite on Kaggle (torch is blocked on the dev laptop). Informational: does not stop the run.
tests = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "-p", "no:cacheprovider"],
                       capture_output=True, text=True)
print(tests.stdout[-3000:])

In [ ]:
# v3 outputs go to results_v3 (MTS_RESULTS_DIR). Retrieval (train-only index), the val-selected localization
# (DR-029) and the train-split retrieval for fine-tuning are reused from committed files.
import json, shutil
os.environ["MTS_RESULTS_DIR"] = RESULTS
for d in ["s0", "s1", "s2", "s3", "finetune"]:
    os.makedirs(f"{RESULTS}/{d}", exist_ok=True)
for f in ["retrieval_val.csv", "retrieval_test.csv", "retrieval_tuning.json"]:
    shutil.copy(f"results/s1/{f}", f"{RESULTS}/s1/{f}")
for split in ["val", "test"]:
    shutil.copy(f"results/s2/localization_{split}_camhidden.csv", f"{RESULTS}/s2/localization_{split}.csv")
    shutil.copy(f"results/s2/localization_{split}_camhidden_summary.json", f"{RESULTS}/s2/localization_{split}_summary.json")
shutil.copy("results/finetune/retrieval_train.csv", f"{RESULTS}/finetune/retrieval_train.csv")

# Resume from a previous session of this notebook, if its output is attached.
for f in glob.glob(f"/kaggle/input/**/{RESULTS}/finetune/train_log.json", recursive=True):
    if json.load(open(f)).get("finished"):
        shutil.copy(f, f"{RESULTS}/finetune/train_log.json")
        print("resumed finished selection log from", f)
        break
prev = glob.glob(f"/kaggle/input/**/{ADAPTER}/adapter_config.json", recursive=True)
if prev and not os.path.exists(ADAPTER):
    shutil.copytree(os.path.dirname(prev[0]), ADAPTER)
    print("resumed adapter from", os.path.dirname(prev[0]))
best = {}
for f in glob.glob(f"/kaggle/input/**/{RESULTS}/s[01]/generations_*.jsonl", recursive=True):
    if os.path.basename(f) not in ("generations_test.jsonl", "generations_val.jsonl"):
        continue
    name = os.path.join(os.path.basename(os.path.dirname(f)), os.path.basename(f))
    n = sum(1 for line in open(f, encoding="utf-8") if line.strip())
    if n > best.get(name, ("", -1))[1]:
        best[name] = (f, n)
for name, (f, n) in sorted(best.items()):
    shutil.copy(f, os.path.join(RESULTS, name))
    print(f"resumed {RESULTS}/{name}: {n} patients")
print("adapter present:", os.path.exists(ADAPTER))

In [ ]:
# Balanced LoRA + projector on the TRAIN split (2 epochs); checkpoint chosen on val generations (DR-032).
LOG = f"{RESULTS}/finetune/train_log.json"
if not (os.path.exists(LOG) and json.load(open(LOG)).get("finished")):
    subprocess.run([sys.executable, "src/pipeline/finetune_vlm.py", "--quant", QUANT, "--recipe", "v3"], check=True)
log = json.load(open(LOG))
for s in log["selection"]:
    print(f"{s['candidate']:>10s}  findings-F1 {s['findings_micro_f1']:.4f}  macro-F1 {s['macro_f1_14']:.4f}  "
          f"distinct {s['distinct_ratio']:.2f}  eligible {s['eligible']}")
print("val loss:", [(h["step"], round(h["val_loss"], 4)) for h in log["history"]])
SELECTED = log["selected"]
print("SELECTED:", SELECTED, f"| elapsed {(time.time() - START) / 3600:.2f} h")

In [ ]:
DEADLINE = START + GEN_BUDGET_H * 3600

def n_done(p):
    return sum(1 for line in open(p, encoding="utf-8") if line.strip()) if os.path.exists(p) else 0

EXPECTED = {f"{RESULTS}/s1/generations_val.jsonl": 366, f"{RESULTS}/s1/generations_test.jsonl": 734,
            f"{RESULTS}/s0/generations_test.jsonl": 734}
if SELECTED == "zero_shot":
    COMPLETE = False
    print("ZERO-SHOT SELECTED on val: no checkpoint beat the base model, so no v3 generation is run.")
else:
    def run(*args):
        cmd = [sys.executable, "src/pipeline/run_generation.py", *args, "--quant", QUANT, "--adapter", ADAPTER,
               "--deadline", str(DEADLINE)]
        print(" ".join(cmd), flush=True)
        subprocess.run(cmd, check=True)

    run("--stage", "s1", "--split", "val", "--samples", "5")
    run("--stage", "s1", "--split", "test", "--samples", "5")
    run("--stage", "s0", "--split", "test")
    COMPLETE = all(n_done(f) == n for f, n in EXPECTED.items())
    for f, n in EXPECTED.items():
        print(f"{f}: {n_done(f)}/{n}")
    print("GENERATION COMPLETE: scoring follows." if COMPLETE else
          "GENERATION NOT FINISHED: scoring skipped. Attach this version's output as an input and Save & Run All again.")

In [ ]:
FAILED = []
if not COMPLETE:
    print("Skipped: no complete v3 generations in this session.")
else:
    def py(script, *args):
        cmd = [sys.executable, script, *args]
        print(" ".join(cmd), flush=True)
        if subprocess.run(cmd).returncode != 0:
            FAILED.append(" ".join(args) or script)

    py("src/evaluation/evaluate_stage.py", f"{RESULTS}/s0/generations_test.jsonl")
    py("src/evaluation/evaluate_stage.py", f"{RESULTS}/s1/generations_test.jsonl")
    py("src/evaluation/evaluate_stage.py", f"{RESULTS}/s1/generations_val.jsonl", "--uncertainty", "--fit-flagger")
    py("src/evaluation/evaluate_stage.py", f"{RESULTS}/s1/generations_test.jsonl", "--uncertainty")
    py("src/pipeline/tune_review_flag.py", "--apply-test")
    py("src/evaluation/build_ablation.py", "--split", "test")
    py("src/evaluation/bootstrap_ci.py", "--split", "test")
    if os.path.exists(f"{RESULTS}/ablation/ablation_test.md"):
        print(open(f"{RESULTS}/ablation/ablation_test.md", encoding="utf-8").read())
print("scoring failures:", FAILED or "none")

In [ ]:
# Always packaged, even if a scoring step failed: everything can be re-scored locally from these files.
import zipfile
out = "/kaggle/working/phase3_v3_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{RESULTS}/**/*", recursive=True)):
        if os.path.isfile(f):
            z.write(f)
    z.writestr(f"{RESULTS}/KAGGLE_RUN.txt",
               subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(out, "| complete:", COMPLETE, "| selected:", SELECTED)